# Document Summarization


This notebook demonstrates an application of long document summarization techniques to a work of literature.

## Install Dependencies


Install the Python packages used by the notebook. Uncomment and run this cell once if they are not already installed in the current environment.

In [1]:
# %pip install -U transformers langchain-ollama langchain-openrouter python-dotenv docling requests

## Provider setup


When `PROVIDER=ollama`, this notebook uses a locally accessible [Ollama](https://github.com/ollama/ollama) server. Pull the exact quantized model in a terminal before running the notebook:

```powershell
ollama pull LiquidAI/lfm2.5-2.6b:q4_k_m
```

To try it interactively from a terminal, run the model and set `num_gpu` before sending a prompt:

```text
ollama run LiquidAI/lfm2.5-2.6b:q4_k_m
>>> /set parameter num_gpu 0
>>> Summarize the main idea of Walden in one paragraph.
```

The Python client below sends `num_gpu=0` with every request, which forces CPU-only inference. While a request is running, `ollama ps` should show `100% CPU` in the `PROCESSOR` column.

## Select your model

The provider is selected from the project's `.env` file. Set `PROVIDER=ollama` to use the local CPU-only model, or `PROVIDER=openrouter` to use the model named by `OPENROUTER_MODEL`. OpenRouter also requires `OPENROUTER_API_KEY`.

```dotenv
# Local CPU inference
PROVIDER=ollama

# Or hosted inference
# PROVIDER=openrouter
# OPENROUTER_API_KEY=your-key
# OPENROUTER_MODEL=provider/model-name
```

For Ollama, a 32K working context is used by default to limit memory use. The model supports up to 128K, but allocating the full context uses considerably more RAM. You can override it with `OLLAMA_NUM_CTX` in `.env`.

In [2]:
import os

import requests
from dotenv import load_dotenv
from langchain_ollama import ChatOllama
from langchain_openrouter import ChatOpenRouter

load_dotenv()

PROVIDER = os.getenv("PROVIDER", "ollama").strip().lower()
OLLAMA_MODEL = os.getenv("OLLAMA_MODEL", "LiquidAI/lfm2.5-2.6b:q4_k_m")
OPENROUTER_MODEL = os.getenv("OPENROUTER_MODEL", "").strip()
TOKENIZER_NAME = os.getenv("TOKENIZER_NAME", "LiquidAI/LFM2.5-2.6B")
OLLAMA_BASE_URL = os.getenv("OLLAMA_HOST", "http://127.0.0.1:11434").rstrip("/")
CONTEXT_LENGTH = int(os.getenv("OLLAMA_NUM_CTX", "32768"))
MAX_OUTPUT_TOKENS = int(os.getenv("MAX_OUTPUT_TOKENS", "2048"))

if PROVIDER == "ollama":
    response = requests.get(f"{OLLAMA_BASE_URL}/api/tags", timeout=10)
    response.raise_for_status()
    installed_models = {item["name"] for item in response.json().get("models", [])}
    if OLLAMA_MODEL not in installed_models:
        raise RuntimeError(f"{OLLAMA_MODEL} is not installed. Run: ollama pull {OLLAMA_MODEL}")

    model = ChatOllama(
        model=OLLAMA_MODEL,
        base_url=OLLAMA_BASE_URL,
        validate_model_on_init=True,
        reasoning=False,
        num_gpu=0,  # Force CPU-only inference.
        num_ctx=CONTEXT_LENGTH,
        num_predict=MAX_OUTPUT_TOKENS,
        temperature=0.1,
        top_k=50,
        repeat_penalty=1.1,
    )
    MODEL_NAME = OLLAMA_MODEL
elif PROVIDER == "openrouter":
    if not os.getenv("OPENROUTER_API_KEY"):
        raise RuntimeError("OPENROUTER_API_KEY is missing from .env")
    if not OPENROUTER_MODEL:
        raise RuntimeError("OPENROUTER_MODEL is missing from .env")

    model = ChatOpenRouter(
        model=OPENROUTER_MODEL,
        max_tokens=MAX_OUTPUT_TOKENS,
        temperature=1.0,
        top_p=0.95,
    )
    MODEL_NAME = OPENROUTER_MODEL
else:
    raise ValueError("PROVIDER must be either 'ollama' or 'openrouter'")

print(f"Provider: {PROVIDER}; model: {MODEL_NAME}")


d:\Workspace\Repository\llm-summarization\.venv\Lib\site-packages\langchain_core\utils\pydantic.py:42: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1 import BaseModel as BaseModelV1


Provider: ollama; model: LiquidAI/lfm2.5-2.6b:q4_k_m


In [3]:
# Confirm the configured model can answer before starting summarization.
sanity_response = model.invoke("Hello!")
print(f"Sanity check response: {sanity_response.content}")


Sanity check response: <think>The user said "Hello!" which is a standard greeting. I should respond politely and offer assistance. There's no complex task or hidden intent here. I'll keep it simple and friendly.</think>Hello! How can I help you today?


## Split a book into chunks

The code in the following sections will fetch H.D. Thoreau's "Walden" from [Project Gutenberg](https://www.gutenberg.org/) for summarization.

We will then chunk the book text so that chunks fit in the context window size of the AI model.

### Count the tokens

Before sending our book chunks to the AI model, it's crucial to understand how much of the model's capacity we're using. Language models typically have a limit on the number of tokens they can process in a single request.

Key points:
- [`LFM2.5-2.6B`](https://huggingface.co/LiquidAI/LFM2.5-2.6B) supports a context window of up to 128K tokens.
- Ollama uses the smaller working context configured above to reduce CPU RAM use.
- Tokenization varies between models. `TOKENIZER_NAME` defaults to LFM2.5-2.6B's tokenizer and can be overridden in `.env` when selecting a different OpenRouter model.

Understanding token count helps us optimize our prompts and ensure we're using the model efficiently.

In [4]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_NAME, trust_remote_code=True)

### Summary of Summaries

Here we use a hierarchical abstractive summarization technique to adapt to the context length of the model. Our approach uses [Docling](https://docling-project.github.io/docling/) to understand the document's structure, chunk the document into text passages, and group the text passages by chapter which we can then summarize. 

In [5]:
import itertools
from collections.abc import Callable, Iterator

from docling.document_converter import DocumentConverter
from docling_core.transforms.chunker.base import BaseChunk
from docling_core.transforms.chunker.hierarchical_chunker import HierarchicalChunker


def chunk_document(source: str, *, dropwhile: Callable[[BaseChunk], bool] = lambda c: False, takewhile: Callable[[BaseChunk], bool] = lambda c: True) -> Iterator[BaseChunk]:
    """Read the document and perform a hierarchical chunking"""
    converter = DocumentConverter()
    chunks = HierarchicalChunker().chunk(converter.convert(source=source).document)
    return itertools.takewhile(takewhile, itertools.dropwhile(dropwhile, chunks))

def merge_chunks(chunks: Iterator[BaseChunk], *, headings: Callable[[BaseChunk], list[str]] = lambda c: c.meta.headings) -> Iterator[dict[str, str]]:
    """Merge chunks having the same headings"""
    prior_headings: list[str] | None = None
    document: dict[str, str] = {}
    doc_id = 0
    for chunk in chunks:
        text = chunk.text.replace('\r\n', '\n')
        current_headings = headings(chunk)
        if prior_headings != current_headings:
            if document:
                yield document
            prior_headings = current_headings
            document = {
                'doc_id': str(doc_id := doc_id + 1),
                'title': " - ".join(current_headings),
                'text': text
            }
        else:
            document['text'] += f"\n\n{text}"
    if document:
        yield document

def chunk_dropwhile(chunk: BaseChunk) -> bool:
    """Ignore front matter prior to the book start"""
    return chunk.meta.headings != ["WALDEN", "Economy"]

def chunk_takewhile(chunk: BaseChunk) -> bool:
    """Ignore remaining chunks once we see this heading"""
    return "ON THE DUTY OF CIVIL DISOBEDIENCE" not in chunk.meta.headings

def chunk_headings(chunk: BaseChunk) -> list[str]:
    """Use the h1 and h2 (chapter) headings"""
    return chunk.meta.headings[:2]

documents: list[dict[str, str]] = list(merge_chunks(
    chunk_document(
        "https://www.gutenberg.org/cache/epub/205/pg205-images.html",
        dropwhile=chunk_dropwhile,
        takewhile=chunk_takewhile,
    ),
    headings=chunk_headings,
))

print(f"{len(documents)} documents created")
print(f"Max document size: {max(len(tokenizer.tokenize(document['text'])) for document in documents)} tokens")

18 documents created
Max document size: 32043 tokens


## Summarize the chunks

Here we define a method to generate a response using a list of documents and a user prompt about those documents. 

The documents are placed directly in standard chat messages so either provider can apply the selected model's native chat template.

In [6]:
from langchain_core.messages import HumanMessage, SystemMessage

SYSTEM_PROMPT = "You are a careful document summarizer. Use only the supplied source documents."

def generate(user_prompt: str, documents: list[dict[str, str]]) -> str:
    """Format source documents as a standard chat request and invoke the selected provider."""
    document_text = "\n\n".join(
        f"<document id='{document['doc_id']}' title='{document['title']}'>\n{document['text']}\n</document>"
        for document in documents
    )
    request_text = f"{user_prompt}\n\nSource documents:\n{document_text}"
    chat = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": request_text},
    ]
    input_tokens = len(tokenizer.apply_chat_template(
        chat, tokenize=True, add_generation_prompt=True
    ))
    if PROVIDER == "ollama" and input_tokens + MAX_OUTPUT_TOKENS > CONTEXT_LENGTH:
        raise ValueError(
            f"Request needs about {input_tokens + MAX_OUTPUT_TOKENS:,} tokens, "
            f"but OLLAMA_NUM_CTX is {CONTEXT_LENGTH:,}. Increase it or use smaller documents."
        )

    print(f"Estimated input size: {input_tokens:,} tokens")
    response = model.invoke([
        SystemMessage(content=SYSTEM_PROMPT),
        HumanMessage(content=request_text),
    ])
    output_text = response.content if isinstance(response.content, str) else str(response.content)
    usage = response.usage_metadata or {}
    actual_input_tokens = usage.get("input_tokens")
    output_tokens = usage.get("output_tokens", len(tokenizer.tokenize(output_text)))
    if actual_input_tokens is not None:
        print(f"Provider-reported input size: {actual_input_tokens:,} tokens")
    print(f"Output size: {output_tokens:,} tokens")

    return output_text

For each chapter, we create a separate summary. CPU-only inference can take a while, especially for the longer chapters. Set `SUMMARIZATION_TESTING=true` before starting Jupyter to process only the first five chapters.

In [7]:
if os.getenv("SUMMARIZATION_TESTING", "false").lower() == "true":
    print("SUMMARIZATION_TESTING is true, only processing the first 5 documents")
    documents = documents[:5] # shorten testing work

user_prompt = """\
Using only the book chapter document, compose a summary of the book chapter.
Your response should only include the summary. Do not provide any further explanation."""

summaries: list[dict[str, str]] = []

for i, document in enumerate(documents, start=1):
    print(f"============================= {document['title']} ({i}/{len(documents)}) =============================")
    output = generate(user_prompt, [document])
    summaries.append({
        'doc_id': document['doc_id'],
        'title': document['title'],
        'text': output,
    })

print("Summary count: " + str(len(summaries)))

SUMMARIZATION_TESTING is true, only processing the first 5 documents
============================= WALDEN - Economy (1/5) =============================
Estimated input size: 2 tokens
Provider-reported input size: 32,122 tokens
Output size: 591 tokens
============================= WALDEN - Where I Lived, and What I Lived For (2/5) =============================
Estimated input size: 2 tokens
Provider-reported input size: 7,670 tokens
Output size: 664 tokens
============================= WALDEN - Reading (3/5) =============================
Estimated input size: 2 tokens
Provider-reported input size: 4,698 tokens
Output size: 640 tokens
============================= WALDEN - Sounds (4/5) =============================
Estimated input size: 2 tokens
Provider-reported input size: 7,481 tokens
Output size: 549 tokens
============================= WALDEN - Solitude (5/5) =============================
Estimated input size: 2 tokens
Provider-reported input size: 4,347 tokens
Output size: 463 toke

## Create the Final Summary

Now we need to summarize the chapter summaries. We prompt the model to create a unified summary of the chapter summaries we previously generated.

In [8]:
import textwrap

user_prompt = """\
Using only the book chapter summary documents, compose a single, unified summary of the book.
Your response should only include the unified summary. Do not provide any further explanation."""

output = generate(user_prompt, summaries)
print(textwrap.fill(output, width=100))

Estimated input size: 2 tokens
Provider-reported input size: 3,059 tokens
Output size: 1,215 tokens
<think>The user wants a single, unified summary of the book using only the provided source
documents. I have four documents: 1. WALDEN - Economy (about living simply at Walden Pond) 2. WALDEN
- Where I Lived, and What I Lived For (about surveying land, Hollowell farm, nature philosophy) 3.
WALDEN - Reading (about deliberate reading, classics, intellectual culture) 4. WALDEN - Sounds
(about trains, nature sounds, isolation)  I need to create a unified summary that covers the main
themes from all these documents while only using information from these sources. The response should
be just the summary with no additional explanation.  Let me combine the key points: - Walden's
experiment in simple living at Walden Pond for two years and two months - Building his own house,
farming, self-sufficiency - Expenses: $28.12½ (house), $14.72½ (farm), $8.74 (food), $8.40¾
(clothing), $2.00 (oil) - Crit

So we have now summarized a document larger than the AI model's context window length by breaking the document down into smaller pieces to summarize and then summarizing those summaries.

Summary from LLM:;
This chapter describes Henry David Thoreau's two-year experiment in simple living at
Walden Pond, where he built his own house from forest materials, cultivated land for food, and
maintained himself through manual labor alone. His annual expenses totaled approximately $61.99¾,
covering construction ($28.12½), farming ($14.72½), food ($8.74), clothing ($8.40¾), and oil
($2.00). Thoreau argues that true necessity consists only of food, shelter, and clothing; all other
possessions are superfluous luxuries that waste time and resources. He concludes that simple living
is not a hardship but a pastime when pursued wisely, as it allows one to live independently without
the burden of unnecessary labor or debt.  The chapter critiques modern society's obsession with
material wealth, excessive work, and social status, suggesting that most people are degraded by
civilization while the poor suffer more. Thoreau emphasizes self-reliance through independence
rather than accumulation, viewing true wealth as freedom from dependence on others' goods. His
experience at the Hollowell farm—an isolated property he purchased but later returned when the owner
changed his mind—illustrates his appreciation for living simply in nature before selling back to the
land.  Thoreau also reflects on surveying land within twelve miles of his home and his interest in
buying farms for temporary residence, though he ultimately found the experience too brief. He
describes his home near a small pond in Concord as ideal for simple living, noting how morning light
on water creates "a lower heaven" and how distant mountains remind him he lives on dry land
surrounded by vast horizons.  In "Where I Lived," Thoreau discusses nature's role as the ultimate
teacher—comparing himself to a loon in the pond or a mullein plant—and argues that Nature's
universal "vegetable, botanic medicines" are his true panacea for well-being. He contrasts this with
the noise and disruption of railroads approaching Concord, which he finds overwhelming yet
ultimately connects him more deeply to nature through its sounds.  In "Reading," Thoreau emphasizes
the importance of deliberate study and engagement with classics—Homer, Æschylus, Virgil—as "the
noblest recorded thoughts of man" that remain undecayed. He criticizes superficial reading in
primers and story books, arguing that true intellectual nourishment requires training like athletic
discipline to master language and its matured form.  Finally, in "Sounds," Thoreau describes the
sensory experience of nature—bell melodies vibrating through the woods like a universal lyre,
whippoorwill vespers at precise times, hooting owls representing human mortality, and distant cow
lowing. He finds beauty in wild birds but notes his lack of domestic animals (no cockerels to crow
or hens to cackle), leaving only squirrels, blue-jays, and occasional whippoorwills in his clearing.
The chapter unifies these themes around Thoreau's conviction that independence through self-reliance
is superior to dependence on others' goods, and that a life of simplicity allows one to focus on
genuine learning, nature, and personal integrity rather than accumulating wealth or status.